In [3]:
import mlflow
import mlflow.pytorch
import pandas as pd
import numpy as np
import os
import warnings
import boto3
from botocore.client import Config
warnings.filterwarnings('ignore')

# ============================================
# НАСТРОЙКА CREDENTIALS ДЛЯ MINIO (S3)
# ============================================
# Настройки из docker-compose
S3_ENDPOINT = "http://localhost:9000"
AWS_ACCESS_KEY = "admin"
AWS_SECRET_KEY = "password"
BUCKET_NAME = "mlflow-bucket"

# Устанавливаем переменные окружения для boto3
os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_KEY
os.environ["MLFLOW_S3_ENDPOINT_URL"] = S3_ENDPOINT
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

# Настройка MLflow
MLFLOW_TRACKING_URI = "http://localhost:5050"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print(f"MLflow URI: {MLFLOW_TRACKING_URI}")
print(f"S3 Endpoint: {S3_ENDPOINT}")
print(f"Bucket: {BUCKET_NAME}")

# ============================================
# ПРОВЕРКА ДОСТУПА К MINIO
# ============================================
print("\n" + "="*60)
print("ПРОВЕРКА ДОСТУПА К MINIO")
print("="*60)

try:
    s3_client = boto3.client(
        's3',
        endpoint_url=S3_ENDPOINT,
        aws_access_key_id=AWS_ACCESS_KEY,
        aws_secret_access_key=AWS_SECRET_KEY,
        config=Config(signature_version='s3v4'),
        region_name='us-east-1'
    )
    
    # Проверяем доступность бакета
    s3_client.head_bucket(Bucket=BUCKET_NAME)
    print(f"✅ Доступ к MinIO успешен!")
    
    # Список объектов в бакете
    objects = s3_client.list_objects_v2(Bucket=BUCKET_NAME, Prefix="mlflow/")
    print(f"✅ Найдено объектов: {objects.get('KeyCount', 0)}")
    
except Exception as e:
    print(f"❌ Ошибка доступа к MinIO: {e}")

# ============================================
# Run ID из предыдущего эксперимента
# ============================================
RUN_ID = "0d5770df8e634cdb92b272fe2f418e78"
print(f"\nRun ID: {RUN_ID}")

# ============================================
# СПОСОБ 1: ЗАГРУЗКА ЧЕРЕЗ runs:/ С ПРАВИЛЬНЫМИ КРЕДАМИ
# ============================================
print("\n" + "="*60)
print("СПОСОБ 1: ЗАГРУЗКА ЧЕРЕЗ runs:/")
print("="*60)

try:
    # Устанавливаем S3 endpoint для MLflow
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    
    # Загружаем модель
    model_uri = f"runs:/{RUN_ID}/nhits_model"
    model = mlflow.pytorch.load_model(model_uri)
    print("✅ Модель успешно загружена через runs:/")
    
    # Получаем информацию о run
    run = mlflow.get_run(RUN_ID)
    print(f"\n📋 ИНФОРМАЦИЯ О МОДЕЛИ:")
    print(f"  • Run ID: {run.info.run_id}")
    print(f"  • Status: {run.info.status}")
    
    # Получаем параметры
    print(f"\n📊 ПАРАМЕТРЫ МОДЕЛИ:")
    for key, value in run.data.params.items():
        print(f"  • {key}: {value}")
    
    # Получаем метрики
    print(f"\n📈 МЕТРИКИ МОДЕЛИ:")
    for key, value in run.data.metrics.items():
        print(f"  • {key}: {float(value):.4f}")
        
except Exception as e:
    print(f"❌ Ошибка загрузки: {e}")

# ============================================
# СПОСОБ 2: ЗАГРУЗКА НАПРЯМУЮ ИЗ S3
# ============================================
print("\n" + "="*60)
print("СПОСОБ 2: ЗАГРУЗКА НАПРЯМУЮ ИЗ S3")
print("="*60)

try:
    # Путь к модели в S3
    s3_model_path = f"s3://{BUCKET_NAME}/mlflow/1/{RUN_ID}/artifacts/nhits_model"
    
    # Скачиваем модель локально
    local_model_path = "./downloaded_model"
    os.makedirs(local_model_path, exist_ok=True)
    
    # Используем boto3 для скачивания
    import subprocess
    result = subprocess.run(
        ["aws", "s3", "cp", "--recursive", s3_model_path, local_model_path, 
         "--endpoint-url", S3_ENDPOINT],
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        print(f"✅ Модель скачана из S3 в {local_model_path}")
        
        # Загружаем модель из локальной директории
        model = mlflow.pytorch.load_model(local_model_path)
        print("✅ Модель успешно загружена из локальной копии")
    else:
        print(f"❌ Ошибка скачивания: {result.stderr}")
        
except Exception as e:
    print(f"❌ Ошибка: {e}")

# ============================================
# СПОСОБ 3: СОЗДАНИЕ ЛОКАЛЬНОЙ КОПИИ ЧЕРЕЗ MLflow ARTIFACT
# ============================================
print("\n" + "="*60)
print("СПОСОБ 3: СОЗДАНИЕ ЛОКАЛЬНОЙ КОПИИ")
print("="*60)

try:
    from mlflow.artifacts import download_artifacts
    
    # Скачиваем артефакты
    local_path = download_artifacts(
        run_id=RUN_ID,
        artifact_path="nhits_model",
        dst_path="./downloaded_artifacts"
    )
    
    print(f"✅ Артефакты скачаны в: {local_path}")
    
    # Загружаем модель
    model = mlflow.pytorch.load_model(local_path)
    print("✅ Модель успешно загружена")
    
except Exception as e:
    print(f"❌ Ошибка скачивания артефактов: {e}")

# ============================================
# ТЕСТОВЫЙ ПРЕДИКТ
# ============================================
print("\n" + "="*60)
print("ТЕСТОВЫЙ ПРЕДИКТ")
print("="*60)

if 'model' in locals():
    import torch
    
    print("✅ Модель загружена, выполняем тестовый предикт...")
    
    # Создаем тестовые данные (60 дней истории)
    test_input = torch.randn(1, 60).float()
    
    # Делаем предсказание
    try:
        model.eval()
        with torch.no_grad():
            prediction = model(test_input)
        print(f"✅ Тестовый предикт выполнен успешно!")
        print(f"   Входная форма: {test_input.shape}")
        print(f"   Выходная форма: {prediction.shape}")
        print(f"   Пример предсказания (первые 5 значений): {prediction[0, :5].tolist()}")
    except Exception as e:
        print(f"❌ Ошибка при предсказании: {e}")
else:
    print("⚠️ Модель не загружена, тестовый предикт пропущен")

# ============================================
# ПРОВЕРКА: ПОЛУЧИТЬ СПИСОК АРТЕФАКТОВ
# ============================================
print("\n" + "="*60)
print("СПИСОК АРТЕФАКТОВ В RUN")
print("="*60)

try:
    from mlflow.tracking import MlflowClient
    client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)
    
    artifacts = client.list_artifacts(RUN_ID)
    print("Доступные артефакты:")
    for artifact in artifacts:
        print(f"  • {artifact.path}")
        
except Exception as e:
    print(f"❌ Ошибка: {e}")

print("\n" + "="*60)
print("✅ ЗАГРУЗКА МОДЕЛИ ЗАВЕРШЕНА")
print("="*60)

MLflow URI: http://localhost:5050
S3 Endpoint: http://localhost:9000
Bucket: mlflow-bucket

ПРОВЕРКА ДОСТУПА К MINIO
✅ Доступ к MinIO успешен!
✅ Найдено объектов: 3

Run ID: 0d5770df8e634cdb92b272fe2f418e78

СПОСОБ 1: ЗАГРУЗКА ЧЕРЕЗ runs:/


❌ Ошибка загрузки: Failed to download artifacts from path 'nhits_model', please ensure that the path is correct.

СПОСОБ 2: ЗАГРУЗКА НАПРЯМУЮ ИЗ S3
❌ Ошибка: [WinError 2] Не удается найти указанный файл

СПОСОБ 3: СОЗДАНИЕ ЛОКАЛЬНОЙ КОПИИ


❌ Ошибка скачивания артефактов: Failed to download artifacts from path 'nhits_model', please ensure that the path is correct.

ТЕСТОВЫЙ ПРЕДИКТ
⚠️ Модель не загружена, тестовый предикт пропущен

СПИСОК АРТЕФАКТОВ В RUN
Доступные артефакты:
  • config.json
  • predictions_sample.csv
  • ticker_metrics.csv

✅ ЗАГРУЗКА МОДЕЛИ ЗАВЕРШЕНА


In [11]:
import torch
import mlflow
import mlflow.pytorch
import os
import pandas as pd
import numpy as np

# ============================================
# НАСТРОЙКА
# ============================================
S3_ENDPOINT = "http://localhost:9000"
AWS_ACCESS_KEY = "admin"
AWS_SECRET_KEY = "password"

os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_KEY
os.environ["MLFLOW_S3_ENDPOINT_URL"] = S3_ENDPOINT
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

MLFLOW_TRACKING_URI = "http://localhost:5050"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# ============================================
# ЗАГРУЗКА МОДЕЛИ
# ============================================
RUN_ID = "bd5a34d36eb24b8b86b1197d6f204879"

print(f"Загрузка модели из MLflow (Run ID: {RUN_ID})...")
model = mlflow.pytorch.load_model(f"runs:/{RUN_ID}/nhits_model")
model.eval()
print("✅ Модель загружена")

# Получаем параметры
from mlflow.tracking import MlflowClient
client = MlflowClient()
run = client.get_run(RUN_ID)

input_size = int(run.data.params.get('input_size', 60))
n_clusters = int(run.data.params.get('n_clusters', 8))

print(f"\n📊 ПАРАМЕТРЫ МОДЕЛИ:")
print(f"  • input_size: {input_size}")
print(f"  • n_clusters: {n_clusters}")

# ============================================
# АНАЛИЗ АРХИТЕКТУРЫ МОДЕЛИ
# ============================================
print("\n" + "="*60)
print("АНАЛИЗ АРХИТЕКТУРЫ МОДЕЛИ")
print("="*60)

# Проверяем, какие экзогенные признаки ожидает модель
print(f"Модель имеет hist_exog_list? {hasattr(model, 'hist_exog_list')}")
if hasattr(model, 'hist_exog_list'):
    print(f"  hist_exog_list: {model.hist_exog_list}")

# Проверяем размерность входных слоев
print(f"\nРазмерность входного слоя энкодера:")
if hasattr(model, 'encoder') and hasattr(model.encoder, '0'):
    first_layer = model.encoder[0]
    print(f"  {first_layer}")
    if hasattr(first_layer, 'in_features'):
        print(f"  in_features: {first_layer.in_features}")

# ============================================
# ПРАВИЛЬНАЯ ФУНКЦИЯ ПРОГНОЗА
# ============================================
def predict_with_nhits(model, input_prices, cluster_onehot):
    """
    Делает прогноз с помощью модели NHITS
    """
    model.eval()
    
    with torch.no_grad():
        # 1. insample_y - исторические цены
        # Размер: (batch_size, input_size, 1)
        insample_y = torch.from_numpy(input_prices).float().reshape(1, -1, 1)
        
        # 2. insample_mask - маска (все данные валидны)
        insample_mask = torch.ones_like(insample_y)
        
        # 3. stat_exog - статические признаки (кластеры)
        # Размер: (batch_size, n_clusters)
        stat_exog = torch.from_numpy(cluster_onehot).float().reshape(1, -1)
        
        # 4. hist_exog - исторические экзогенные признаки
        # Для каждого временного шага нужны одинаковые кластерные признаки
        # Размер: (batch_size, input_size, n_clusters)
        hist_exog = stat_exog.unsqueeze(1).repeat(1, input_size, 1)
        
        # 5. futr_exog - будущие экзогенные признаки (для прогноза)
        # Размер: (batch_size, horizon, n_clusters)
        horizon = 30
        futr_exog = stat_exog.unsqueeze(1).repeat(1, horizon, 1)
        
        # Формируем входной словарь
        batch = {
            'insample_y': insample_y,
            'insample_mask': insample_mask,
            'stat_exog': stat_exog,
            'hist_exog': hist_exog,
            'futr_exog': futr_exog
        }
        
        # Делаем прогноз
        output = model(batch)
        predictions = output.squeeze().cpu().numpy()
    
    return predictions

# ============================================
# РЕАЛЬНЫЙ ПРИМЕР: ПРОГНОЗ ДЛЯ AAPL
# ============================================
print("\n" + "="*60)
print("РЕАЛЬНЫЙ ПРИМЕР: ПРОГНОЗ ДЛЯ AAPL")
print("="*60)

# Загружаем данные
df = pd.read_csv('prices_all.csv')
df['date'] = pd.to_datetime(df['date'])

# Берем последние input_size дней AAPL
aapl_data = df[df['Ticker'] == 'AAPL'].sort_values('date').tail(input_size)
last_prices = aapl_data['Close'].values

if len(last_prices) < input_size:
    print(f"⚠️ Недостаточно данных для AAPL: {len(last_prices)} < {input_size}")
    # Берем все доступные
    input_size = len(last_prices)

print(f"Последние {input_size} цен AAPL:")
print(f"  Первые 5: {last_prices[:5]}")
print(f"  Последняя цена: ${last_prices[-1]:.2f}")

# Загружаем кластеры
cluster_df = pd.read_csv('cluster_fullstart_assignments.csv')
aapl_cluster = cluster_df[cluster_df['Company'] == 'AAPL']['Cluster'].values[0]
print(f"\nКластер AAPL: {aapl_cluster}")

# Создаем one-hot вектор кластера
cluster_onehot = np.zeros(n_clusters)
cluster_onehot[aapl_cluster] = 1
print(f"One-hot кластера: {cluster_onehot}")

# Делаем прогноз
print("\n📈 ГЕНЕРАЦИЯ ПРОГНОЗА...")
try:
    predictions = predict_with_nhits(model, last_prices, cluster_onehot)
    
    print(f"\n📈 ПРОГНОЗ AAPL НА СЛЕДУЮЩИЕ 30 ДНЕЙ:")
    for i, val in enumerate(predictions):
        print(f"  День {i+1}: ${val:.2f}")
    
    # Статистика прогноза
    print(f"\n📊 СТАТИСТИКА ПРОГНОЗА:")
    print(f"  • Текущая цена: ${last_prices[-1]:.2f}")
    print(f"  • Минимум: ${min(predictions):.2f}")
    print(f"  • Максимум: ${max(predictions):.2f}")
    print(f"  • Среднее: ${np.mean(predictions):.2f}")
    print(f"  • Изменение: ${predictions[-1] - last_prices[-1]:.2f}")
    print(f"  • Изменение (%): {((predictions[-1] - last_prices[-1]) / last_prices[-1] * 100):.2f}%")
    
    # Сохраняем прогноз
    forecast_df = pd.DataFrame({
        'day': range(1, len(predictions) + 1),
        'predicted_price': predictions
    })
    forecast_df.to_csv('aapl_forecast.csv', index=False)
    print("\n✅ Прогноз сохранен в 'aapl_forecast.csv'")
    
except Exception as e:
    print(f"❌ Ошибка: {e}")
    import traceback
    traceback.print_exc()

# ============================================
# ПРИМЕР ДЛЯ НЕСКОЛЬКИХ ТИКЕРОВ
# ============================================
print("\n" + "="*60)
print("ПРИМЕР ДЛЯ НЕСКОЛЬКИХ ТИКЕРОВ")
print("="*60)

sample_tickers = ['AAPL', 'MSFT', 'GOOG', 'AMZN', 'NVDA']

results = []
for ticker in sample_tickers:
    # Получаем последние цены
    ticker_data = df[df['Ticker'] == ticker].sort_values('date').tail(input_size)
    if len(ticker_data) >= 60:
        prices = ticker_data['Close'].values[-60:]
        
        # Получаем кластер
        ticker_cluster = cluster_df[cluster_df['Company'] == ticker]['Cluster'].values
        if len(ticker_cluster) > 0:
            cluster_id = ticker_cluster[0]
            cluster_oh = np.zeros(n_clusters)
            cluster_oh[cluster_id] = 1
            
            try:
                pred = predict_with_nhits(model, prices, cluster_oh)
                
                results.append({
                    'Ticker': ticker,
                    'Current_Price': prices[-1],
                    'Forecast_30d': pred[-1],
                    'Change_%': ((pred[-1] - prices[-1]) / prices[-1] * 100),
                    'Min_30d': min(pred),
                    'Max_30d': max(pred),
                    'Cluster': cluster_id
                })
                
                print(f"\n{ticker} (кластер {cluster_id}):")
                print(f"  Текущая: ${prices[-1]:.2f}")
                print(f"  Прогноз: ${pred[-1]:.2f}")
                print(f"  Изменение: {((pred[-1] - prices[-1]) / prices[-1] * 100):+.2f}%")
                print(f"  Диапазон: ${min(pred):.2f} - ${max(pred):.2f}")
                
            except Exception as e:
                print(f"Ошибка для {ticker}: {e}")

# Сохраняем результаты
if results:
    results_df = pd.DataFrame(results)
    results_df.to_csv('forecast_summary.csv', index=False)
    print("\n✅ Сводка прогнозов сохранена в 'forecast_summary.csv'")

print("\n" + "="*60)
print("✅ ГОТОВО!")
print("="*60)

Загрузка модели из MLflow (Run ID: bd5a34d36eb24b8b86b1197d6f204879)...


✅ Модель загружена

📊 ПАРАМЕТРЫ МОДЕЛИ:
  • input_size: 60
  • n_clusters: 8

АНАЛИЗ АРХИТЕКТУРЫ МОДЕЛИ
Модель имеет hist_exog_list? True
  hist_exog_list: ['cluster_id', 'cluster_0', 'cluster_1', 'cluster_2', 'cluster_3', 'cluster_4', 'cluster_5', 'cluster_6']

Размерность входного слоя энкодера:

РЕАЛЬНЫЙ ПРИМЕР: ПРОГНОЗ ДЛЯ AAPL
Последние 60 цен AAPL:
  Первые 5: [255.21260071 254.18359375 254.38340759 255.2026062  256.88098145]
  Последняя цена: $273.67

Кластер AAPL: 4
One-hot кластера: [0. 0. 0. 0. 1. 0. 0. 0.]

📈 ГЕНЕРАЦИЯ ПРОГНОЗА...

📈 ПРОГНОЗ AAPL НА СЛЕДУЮЩИЕ 30 ДНЕЙ:
  День 1: $277.01
  День 2: $280.98
  День 3: $283.61
  День 4: $283.99
  День 5: $290.60
  День 6: $290.13
  День 7: $290.60
  День 8: $296.35
  День 9: $303.13
  День 10: $305.03
  День 11: $309.63
  День 12: $310.52
  День 13: $322.20
  День 14: $324.74
  День 15: $328.97
  День 16: $329.04
  День 17: $334.62
  День 18: $338.08
  День 19: $345.03
  День 20: $348.48
  День 21: $354.48
  День 22: $362.43
  Ден